In [36]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nih-chest-xrays/data")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/nih-chest-xrays/data


In [37]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import densenet121, DenseNet121_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

In [38]:
import os

DATA_DIR = "/kaggle/input/datasets/nih-chest-xrays/data"

CSV_PATH = os.path.join(
    DATA_DIR,
    "Data_Entry_2017.csv"
)

IMAGE_DIRS = [
    os.path.join(DATA_DIR, f"images_{i:03d}", "images")
    for i in range(1, 13)
]

print("CSV:", CSV_PATH)

for directory in IMAGE_DIRS:
    print(directory, "->", os.path.exists(directory))

CSV: /kaggle/input/datasets/nih-chest-xrays/data/Data_Entry_2017.csv
/kaggle/input/datasets/nih-chest-xrays/data/images_001/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_002/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_003/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_004/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_005/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_006/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_007/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_008/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_009/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_010/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_011/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_012/images -> True


In [39]:
image_paths = {}

for image_dir in IMAGE_DIRS:
    for image_name in os.listdir(image_dir):
        image_paths[image_name] = os.path.join(
            image_dir,
            image_name
        )

print("Total images found:", len(image_paths))

Total images found: 112120


In [40]:
import pandas as pd

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (112120, 12)


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


In [41]:
missing_images = [
    image_name
    for image_name in df["Image Index"]
    if image_name not in image_paths
]

print("Missing images:", len(missing_images))

if len(missing_images) > 0:
    print(missing_images[:10])

Missing images: 0


In [42]:
from torch.utils.data import Dataset
from PIL import Image
import torch
import numpy as np


class ChestXrayDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_paths,
        transform=None
    ):

        self.dataframe = dataframe.reset_index(drop=True)
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_name = row["Image Index"]

        image_path = self.image_paths[image_name]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(
            row[diseases].values.astype(np.float32)
        )

        return image, labels

In [43]:
classes = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Effusion",
    "Emphysema",
    "Fibrosis",
    "Hernia",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pleural_Thickening",
    "Pneumonia",
    "Pneumothorax",
    "No Finding"
]

print("Number of classes:", len(classes))

Number of classes: 15


In [44]:
for class_name in classes:
    df[class_name] = df["Finding Labels"].apply(
        lambda x: 1 if class_name in x.split("|") else 0
    )

In [45]:
print(df[classes].sum().sort_values())

Hernia                  227
Pneumonia              1431
Fibrosis               1686
Edema                  2303
Emphysema              2516
Cardiomegaly           2776
Pleural_Thickening     3385
Consolidation          4667
Pneumothorax           5302
Mass                   5782
Nodule                 6331
Atelectasis           11559
Effusion              13317
Infiltration          19894
No Finding            60361
dtype: int64


In [46]:
from sklearn.model_selection import train_test_split

patients = df["Patient ID"].unique()

train_patients, temp_patients = train_test_split(
    patients,
    test_size=0.20,
    random_state=42
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=42
)

train_df = df[df["Patient ID"].isin(train_patients)].copy()
val_df = df[df["Patient ID"].isin(val_patients)].copy()
test_df = df[df["Patient ID"].isin(test_patients)].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 89826
Validation: 10930
Test: 11364


In [47]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [48]:
from torch.utils.data import Dataset
from PIL import Image
import torch
import numpy as np

class ChestXrayDataset(Dataset):

    def __init__(self, dataframe, image_paths, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_name = row["Image Index"]
        image_path = self.image_paths[image_name]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(
            row[classes].values.astype(np.float32)
        )

        return image, labels

In [49]:
train_dataset = ChestXrayDataset(
    train_df,
    image_paths,
    train_transform
)

val_dataset = ChestXrayDataset(
    val_df,
    image_paths,
    val_transform
)

test_dataset = ChestXrayDataset(
    test_df,
    image_paths,
    val_transform
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 89826
Validation: 10930
Test: 11364


In [50]:
image, label = train_dataset[0]

print("Image shape:", image.shape)
print("Label shape:", label.shape)
print("Labels:", label)

Image shape: torch.Size([3, 224, 224])
Label shape: torch.Size([15])
Labels: tensor([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


In [51]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("DataLoaders recreated successfully.")
images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)

DataLoaders recreated successfully.
Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32, 15])


In [52]:
images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)

Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32, 15])


# Load EfficientNet-B0

In [53]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

weights = EfficientNet_B0_Weights.DEFAULT

efficientnet = efficientnet_b0(
    weights=weights
)

print(efficientnet)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [54]:
num_features = efficientnet.classifier[1].in_features

efficientnet.classifier[1] = nn.Linear(
    num_features,
    len(classes)
)

efficientnet = efficientnet.to(device)

print(efficientnet.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=15, bias=True)
)


In [55]:
positive_counts = train_df[classes].sum().values
negative_counts = len(train_df) - positive_counts

pos_weights = negative_counts / positive_counts

pos_weights = torch.tensor(
    pos_weights,
    dtype=torch.float32
).to(device)

criterion_eff = nn.BCEWithLogitsLoss(
    pos_weight=pos_weights
)

In [56]:
optimizer_eff = torch.optim.AdamW(
    efficientnet.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [57]:
scheduler_eff = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_eff,
    mode="max",
    factor=0.5,
    patience=2
)

In [58]:
from torch.amp import autocast, GradScaler

scaler_eff = GradScaler("cuda")

In [59]:
from tqdm.auto import tqdm

def train_effnet(
    model,
    loader,
    criterion,
    optimizer,
    scaler
):

    model.train()

    running_loss = 0.0

    progress = tqdm(
        loader,
        desc="EfficientNet Training"
    )

    for images, labels in progress:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with autocast("cuda"):

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return running_loss / len(loader)

In [60]:
from sklearn.metrics import roc_auc_score

def validate_effnet(
    model,
    loader,
    criterion
):

    model.eval()

    running_loss = 0.0

    all_labels = []
    all_probs = []

    with torch.no_grad():

        for images, labels in tqdm(
            loader,
            desc="EfficientNet Validation"
        ):

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            with autocast("cuda"):

                outputs = model(images)

                loss = criterion(
                    outputs,
                    labels
                )

            probabilities = torch.sigmoid(
                outputs
            )

            running_loss += loss.item()

            all_labels.append(
                labels.cpu().numpy()
            )

            all_probs.append(
                probabilities.cpu().numpy()
            )

    all_labels = np.concatenate(
        all_labels
    )

    all_probs = np.concatenate(
        all_probs
    )

    auc_scores = []

    for i in range(len(classes)):

        auc = roc_auc_score(
            all_labels[:, i],
            all_probs[:, i]
        )

        auc_scores.append(auc)

    mean_auc = np.mean(auc_scores)

    return (
        running_loss / len(loader),
        mean_auc,
        all_labels,
        all_probs
    )

In [61]:
'''train_loss_eff = train_effnet(
    efficientnet,
    train_loader,
    criterion_eff,
    optimizer_eff,
    scaler_eff
)

val_loss_eff, val_auc_eff, _, _ = validate_effnet(
    efficientnet,
    val_loader,
    criterion_eff
)

print()
print(f"Train Loss : {train_loss_eff:.4f}")
print(f"Val Loss   : {val_loss_eff:.4f}")
print(f"Val ROC-AUC: {val_auc_eff:.4f}")'''

'train_loss_eff = train_effnet(\n    efficientnet,\n    train_loader,\n    criterion_eff,\n    optimizer_eff,\n    scaler_eff\n)\n\nval_loss_eff, val_auc_eff, _, _ = validate_effnet(\n    efficientnet,\n    val_loader,\n    criterion_eff\n)\n\nprint()\nprint(f"Train Loss : {train_loss_eff:.4f}")\nprint(f"Val Loss   : {val_loss_eff:.4f}")\nprint(f"Val ROC-AUC: {val_auc_eff:.4f}")'

In [62]:
EPOCHS_EFF = 8

best_auc_eff = 0.0
patience_eff = 3
epochs_without_improvement_eff = 0

history_eff = {
    "train_loss": [],
    "val_loss": [],
    "val_auc": []
}

BEST_EFF_PATH = "/kaggle/working/best_efficientnet_b0.pth"

print("EfficientNet-B0 training ready.")

EfficientNet-B0 training ready.


In [63]:
'''for epoch in range(EPOCHS_EFF):

    print("\n" + "=" * 60)
    print(f"EfficientNet-B0 - Epoch {epoch + 1}/{EPOCHS_EFF}")
    print("=" * 60)

    # -------------------------
    # Training
    # -------------------------

    train_loss = train_effnet(
        efficientnet,
        train_loader,
        criterion_eff,
        optimizer_eff,
        scaler_eff
    )

    # -------------------------
    # Validation
    # -------------------------

    val_loss, val_auc, _, _ = validate_effnet(
        efficientnet,
        val_loader,
        criterion_eff
    )

    # -------------------------
    # Scheduler
    # -------------------------

    scheduler_eff.step(val_auc)

    # -------------------------
    # Save history
    # -------------------------

    history_eff["train_loss"].append(train_loss)
    history_eff["val_loss"].append(val_loss)
    history_eff["val_auc"].append(val_auc)

    print(f"\nTrain Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val ROC-AUC: {val_auc:.4f}")

    # -------------------------
    # Save best model
    # -------------------------

    if val_auc > best_auc_eff:

        best_auc_eff = val_auc
        epochs_without_improvement_eff = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": efficientnet.state_dict(),
                "optimizer_state_dict": optimizer_eff.state_dict(),
                "val_auc": val_auc,
                "classes": classes
            },
            BEST_EFF_PATH
        )

        print("🔥 BEST EFFICIENTNET MODEL SAVED!")

    else:

        epochs_without_improvement_eff += 1

        print(
            f"No improvement: "
            f"{epochs_without_improvement_eff}/{patience_eff}"
        )

        if epochs_without_improvement_eff >= patience_eff:

            print("\n⛔ Early stopping.")
            break'''

'for epoch in range(EPOCHS_EFF):\n\n    print("\n" + "=" * 60)\n    print(f"EfficientNet-B0 - Epoch {epoch + 1}/{EPOCHS_EFF}")\n    print("=" * 60)\n\n    # -------------------------\n    # Training\n    # -------------------------\n\n    train_loss = train_effnet(\n        efficientnet,\n        train_loader,\n        criterion_eff,\n        optimizer_eff,\n        scaler_eff\n    )\n\n    # -------------------------\n    # Validation\n    # -------------------------\n\n    val_loss, val_auc, _, _ = validate_effnet(\n        efficientnet,\n        val_loader,\n        criterion_eff\n    )\n\n    # -------------------------\n    # Scheduler\n    # -------------------------\n\n    scheduler_eff.step(val_auc)\n\n    # -------------------------\n    # Save history\n    # -------------------------\n\n    history_eff["train_loss"].append(train_loss)\n    history_eff["val_loss"].append(val_loss)\n    history_eff["val_auc"].append(val_auc)\n\n    print(f"\nTrain Loss : {train_loss:.4f}")\n   

In [64]:
import os

MODEL_DIR = "/kaggle/input/models/andreasheva/best-efficientnet-b0/pytorch/default/1"

print(os.listdir(MODEL_DIR))

['best_efficientnet_b0.pth']


In [65]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

efficientnet = efficientnet_b0(weights=None)

# Replace classifier for your 15 classes
efficientnet.classifier[1] = nn.Linear(
    efficientnet.classifier[1].in_features,
    15
)

efficientnet = efficientnet.to(device)

print(efficientnet.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=15, bias=True)
)


In [66]:
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    print(list(checkpoint.keys())[:30])

Checkpoint type: <class 'dict'>

Checkpoint keys:
['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_auc', 'classes']


In [67]:
if isinstance(checkpoint, dict):
    for key, value in checkpoint.items():
        print(
            key,
            type(value),
            getattr(value, "shape", "")
        )

epoch <class 'int'> 
model_state_dict <class 'collections.OrderedDict'> 
optimizer_state_dict <class 'dict'> 
val_auc <class 'numpy.float64'> ()
classes <class 'list'> 


In [68]:
efficientnet.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [69]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

efficientnet = efficientnet_b0(weights=None)

# Your checkpoint has 15 classes
efficientnet.classifier[1] = nn.Linear(
    efficientnet.classifier[1].in_features,
    15
)

efficientnet = efficientnet.to(device)

print("✅ EfficientNet-B0 created")

✅ EfficientNet-B0 created


In [70]:
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

efficientnet.load_state_dict(
    checkpoint["model_state_dict"]
)

print("✅ EfficientNet-B0 checkpoint loaded!")
print("Previous epoch:", checkpoint["epoch"])
print("Previous Val AUC:", checkpoint["val_auc"])
print("Classes:", checkpoint["classes"])

✅ EfficientNet-B0 checkpoint loaded!
Previous epoch: 6
Previous Val AUC: 0.8227186674690774
Classes: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax', 'No Finding']


In [71]:
print(
    "Classifier:",
    efficientnet.classifier
)

print(
    "Output classes:",
    efficientnet.classifier[1].out_features
)

Classifier: Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=15, bias=True)
)
Output classes: 15


In [72]:
for param in efficientnet.parameters():
    param.requires_grad = False

In [73]:
for param in efficientnet.features[-3:].parameters():
    param.requires_grad = True

In [74]:
for param in efficientnet.classifier.parameters():
    param.requires_grad = True

In [75]:
criterion_eff_ft = nn.BCEWithLogitsLoss(
    pos_weight=pos_weights.to(device)
)

In [77]:
optimizer_eff_ft = torch.optim.AdamW(
    [
        {
            "params": efficientnet.features[-3:].parameters(),
            "lr": 1e-5
        },
        {
            "params": efficientnet.classifier.parameters(),
            "lr": 5e-5
        }
    ],
    weight_decay=1e-4
)

In [78]:
scheduler_eff_ft = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_eff_ft,
    mode="max",
    factor=0.5,
    patience=2,
    min_lr=1e-7
)

In [79]:
from torch.amp import autocast, GradScaler

scaler_eff_ft = GradScaler("cuda")

In [80]:
efficientnet.eval()

images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device).float()

with torch.no_grad():
    with autocast("cuda"):
        outputs = efficientnet(images)

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Outputs:", outputs.shape)

Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32, 15])
Outputs: torch.Size([32, 15])


In [81]:
from sklearn.metrics import roc_auc_score
import numpy as np
import time

# =========================
# FINE-TUNING SETUP
# =========================

# Freeze everything
for param in efficientnet.parameters():
    param.requires_grad = False

# Unfreeze final 3 blocks
for param in efficientnet.features[-3:].parameters():
    param.requires_grad = True

# Unfreeze classifier
for param in efficientnet.classifier.parameters():
    param.requires_grad = True


criterion_eff_ft = nn.BCEWithLogitsLoss(
    pos_weight=pos_weights.to(device)
)

optimizer_eff_ft = torch.optim.AdamW(
    [
        {
            "params": efficientnet.features[-3:].parameters(),
            "lr": 1e-5
        },
        {
            "params": efficientnet.classifier.parameters(),
            "lr": 5e-5
        }
    ],
    weight_decay=1e-4
)

scheduler_eff_ft = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_eff_ft,
    mode="max",
    factor=0.5,
    patience=2,
    min_lr=1e-7
)

scaler_eff_ft = torch.amp.GradScaler("cuda")

print("✅ EfficientNet fine-tuning setup ready!")

✅ EfficientNet fine-tuning setup ready!


In [ ]:
num_epochs = 15
best_val_auc = 0.0

for epoch in range(num_epochs):

    epoch_start = time.time()

    # =========================
    # TRAIN
    # =========================
    efficientnet.train()

    train_loss = 0.0
    train_total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float()

        optimizer_eff_ft.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda"):
            outputs = efficientnet(images)
            loss = criterion_eff_ft(outputs, labels)

        scaler_eff_ft.scale(loss).backward()
        scaler_eff_ft.step(optimizer_eff_ft)
        scaler_eff_ft.update()

        train_loss += loss.item() * images.size(0)
        train_total += images.size(0)

        # Progress
        if batch_idx % 200 == 0:
            print(
                f"Epoch {epoch+1}/{num_epochs} | "
                f"Batch {batch_idx}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    train_loss /= train_total


    # =========================
    # VALIDATION
    # =========================
    efficientnet.eval()

    val_loss = 0.0
    val_total = 0

    all_probs = []
    all_labels = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).float()

            with torch.amp.autocast("cuda"):
                outputs = efficientnet(images)
                loss = criterion_eff_ft(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_total += images.size(0)

            probs = torch.sigmoid(outputs)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    val_loss /= val_total

    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # =========================
    # AUC
    # =========================
    try:
        val_auc = roc_auc_score(
            all_labels,
            all_probs,
            average="macro"
        )
    except ValueError:
        val_auc = 0.0

    # Scheduler uses AUC
    scheduler_eff_ft.step(val_auc)


    # =========================
    # SAVE BEST
    # =========================
    if val_auc > best_val_auc:

        best_val_auc = val_auc

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": efficientnet.state_dict(),
                "optimizer_state_dict": optimizer_eff_ft.state_dict(),
                "val_auc": val_auc,
                "classes": checkpoint["classes"]
            },
            "best_efficientnet_b0_finetuned.pth"
        )

        print("🔥 BEST MODEL SAVED!")


    # =========================
    # EPOCH RESULT
    # =========================
    print(
        f"\n{'='*60}\n"
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Val Loss: {val_loss:.4f} "
        f"| Val AUC: {val_auc:.4f} "
        f"({val_auc*100:.2f}%) "
        f"| LR: {optimizer_eff_ft.param_groups[0]['lr']:.2e} "
        f"| Time: {(time.time()-epoch_start)/60:.1f} min\n"
        f"{'='*60}"
    )


print(
    f"\n🏆 BEST VALIDATION AUC: "
    f"{best_val_auc:.4f} "
    f"({best_val_auc*100:.2f}%)"
)

Epoch 1/15 | Batch 0/2808 | Loss: 0.6492
Epoch 1/15 | Batch 200/2808 | Loss: 0.5637
Epoch 1/15 | Batch 400/2808 | Loss: 0.6347
Epoch 1/15 | Batch 600/2808 | Loss: 1.1279
Epoch 1/15 | Batch 800/2808 | Loss: 0.6140
Epoch 1/15 | Batch 1000/2808 | Loss: 0.8199
Epoch 1/15 | Batch 1200/2808 | Loss: 0.6476
Epoch 1/15 | Batch 1400/2808 | Loss: 0.5073
Epoch 1/15 | Batch 1600/2808 | Loss: 0.6886
Epoch 1/15 | Batch 1800/2808 | Loss: 0.5785
Epoch 1/15 | Batch 2000/2808 | Loss: 0.6718
Epoch 1/15 | Batch 2200/2808 | Loss: 0.9838
Epoch 1/15 | Batch 2400/2808 | Loss: 1.0723
Epoch 1/15 | Batch 2600/2808 | Loss: 0.6769
Epoch 1/15 | Batch 2800/2808 | Loss: 0.6067
🔥 BEST MODEL SAVED!

Epoch [1/15] | Train Loss: 0.7271 | Val Loss: 1.0415 | Val AUC: 0.8258 (82.58%) | LR: 1.00e-05 | Time: 58.7 min
Epoch 2/15 | Batch 0/2808 | Loss: 0.5657
Epoch 2/15 | Batch 200/2808 | Loss: 0.6902
Epoch 2/15 | Batch 400/2808 | Loss: 0.9220
Epoch 2/15 | Batch 600/2808 | Loss: 0.6439
Epoch 2/15 | Batch 800/2808 | Loss: 0.8314
E